# 📊 Exploratory Data Analysis (EDA 04) - Products Master Dataset (`Products.csv`)
**Mục tiêu**: Phân tích danh mục sản phẩm (`Products.csv`), thương hiệu (`Brand`), danh mục ngành hàng (`Category`), giá niêm yết gốc (`Base_Price`), kiểm tra toàn vẹn định danh khóa chính (`Product_ID`) và đánh giá yêu cầu mô hình hóa Slowly Changing Dimension (SCD Type 2).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập cấu hình hiển thị biểu đồ
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. Tải Dữ Liệu & Khai Thác Cấu Trúc (Schema Audit)

In [2]:
file_path = '../rawdata_test_ae/Products.csv'
df_prod = pd.read_csv(file_path)

print(f"===> Kích thước danh mục sản phẩm Products: {df_prod.shape[0]} dòng, {df_prod.shape[1]} cột")
display(df_prod)

📌 **Nhận xét Cấu trúc Danh mục Sản phẩm (`Products.csv`)**:
- Tập dữ liệu thô gồm **10 bản ghi danh mục sản phẩm** chính đại diện cho các mặt hàng công nghệ chủ lực kinh doanh tại CellphoneS.
- Các thuộc tính gồm 5 cột: `Product_ID` (Mã định danh sản phẩm), `Product_Name` (Tên sản phẩm), `Brand` (Thương hiệu sản xuất), `Category` (Danh mục ngành hàng) và `Base_Price` (Giá niêm yết gốc).

## 2. Kiểm Tra Trùng Lặp Khóa Chính & Dữ Liệu Khuyết Thiếu (Audit)

In [3]:
print("=== BẢNG KIỂM TRA TRÙNG LẶP & KHUYẾT THIẾU ===")
print(f"Số lượng Product_ID duy nhất (Unique): {df_prod['Product_ID'].nunique()} / Tổng số dòng: {len(df_prod)}")
print(f"Số lượng bản ghi trùng lặp 100% (Full Duplicates): {df_prod.duplicated().sum()} dòng")

null_df = pd.DataFrame({
    'Kiểu_Dữ_Liệu': df_prod.dtypes.astype(str),
    'Số_Lượng_Null': df_prod.isnull().sum(),
    'Tỷ_Lệ_Null (%)': (df_prod.isnull().sum() / len(df_prod)) * 100
})
display(null_df)

📌 **Nhận xét & Kết luận Kiểm tra Khóa & Dữ liệu Khuyết thiếu**:
- **Độ toàn vẹn định danh**: Cột `Product_ID` chứa 10 giá trị duy nhất (`P001` đến `P010`), đạt **0 bản ghi trùng lặp (0 Duplicates)**. Đủ điều kiện làm Khóa tự nhiên (Natural Business Key) cho bảng Dimension Sản phẩm.
- **Dữ liệu khuyết thiếu**: Đạt **0 giá trị Null** trên tất cả 5 thuộc tính, đảm bảo chất lượng dữ liệu sạch 100% để xây dựng mô hình Data Warehouse.

## 3. Phân Tích Cơ Cấu Thương Hiệu, Ngành Hàng & Phân Phối Giá Niêm Yết (`Base_Price`)

In [4]:
# Thống kê mô tả giá niêm yết Base_Price
print("=== BẢNG THỐNG KÊ MÔ TẢ GIÁ NIÊM YẾT BASE_PRICE ===")
display(df_prod[['Base_Price']].describe().T)

# Trực quan hóa giá niêm yết theo sản phẩm và cơ cấu danh mục
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

df_sorted = df_prod.sort_values('Base_Price', ascending=True)
sns.barplot(data=df_sorted, y='Product_Name', x=df_sorted['Base_Price']/1e6, palette='magma', ax=axes[0])
axes[0].set_title('Giá Niêm Yết Gốc Theo Sản Phẩm (Triệu VND)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Giá Niêm Yết (Triệu VND)')
axes[0].set_ylabel('Tên Sản Phẩm')

sns.countplot(data=df_prod, x='Category', palette='Set2', ax=axes[1], order=df_prod['Category'].value_counts().index)
axes[1].set_title('Phân Bổ Sản Phẩm Theo Danh Mục (Category)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Danh Mục Sản Phẩm')
axes[1].set_ylabel('Số Lượng SKU')

plt.tight_layout()
plt.show()

📌 **Nhận xét Phân bổ Ngành hàng & Biên độ Giá Niêm Yết (`Base_Price`)**:
- **Phân bổ Ngành hàng (`Category`)**: `Smartphone` chiếm tỷ trọng chủ đạo với **5 SKU (50.0%)**, tiếp theo là `Accessories` với **2 SKU (20.0%)**, `Laptop`, `Tablet` và `Smartwatch` mỗi ngành hàng đóng góp 1 SKU.
- **Cơ cấu Thương hiệu (`Brand`)**: `Apple` áp đảo với 5 SKU (iPhone, iPad, MacBook, AirPods, Apple Watch), `Samsung` có 2 SKU (Galaxy S24 Ultra, Z Fold 5), `Xiaomi`, `Oppo`, `Marshall` mỗi hãng 1 SKU.
- **Phân phối Giá niêm yết (`Base_Price`)**: Giá trị trung bình đạt **21.1 triệu VND**, trung vị đạt **24.5 triệu VND**. Mức giá cao nhất là **40.0 triệu VND** (`P009` - Samsung Galaxy Z Fold 5) và thấp nhất là **6.0 triệu VND** (`P005` - AirPods Pro 2).

## 4. Phân Tích Yêu Cầu Mô Hình Hóa Data Warehouse & Thiết Kế Slowly Changing Dimension (SCD Type 2)

In [5]:
# So sánh biến động giá bán thực tế (Unit_Price trong Transactions) vs Giá gốc (Base_Price)
file_txns = '../rawdata_test_ae/Transactions.csv'
df_txns = pd.read_csv(file_txns)

price_diff = pd.merge(df_txns[['Product_ID', 'Unit_Price', 'Date']].dropna(), df_prod[['Product_ID', 'Product_Name', 'Base_Price']], on='Product_ID', how='left')
price_diff['Price_Variance'] = price_diff['Unit_Price'] - price_diff['Base_Price']

print("=== MẪU BIẾN ĐỘNG GIÁ BÁN THỰC TẾ THEO KHUYẾN MÃI (PROMOTION VARIANCE) ===")
display(price_diff[price_diff['Price_Variance'] != 0][['Date', 'Product_ID', 'Product_Name', 'Base_Price', 'Unit_Price', 'Price_Variance']].head(10))

📌 **Nhận xét & Lập luận Kỹ thuật SCD Type 2 cho Bảng Dimension Sản phẩm (`dim_products_scd2`)**:
- **Đặt vấn đề nghiệp vụ**: Trong chuỗi bán lẻ CellphoneS, đơn giá bán của sản phẩm liên tục thay đổi theo các chương trình khuyến mãi (Promotion), chính sách trợ giá sàn thương mại và các đợt điều chỉnh giá niêm yết hãng.
- **Lập luận chọn SCD Type 2**: Nếu áp dụng **SCD Type 1 (Ghi đè - Overwrite)**, khi giá sản phẩm thay đổi, hệ thống sẽ ghi đè giá mới lên giá cũ, dẫn đến sai lệch hoàn toàn kết quả doanh thu lịch sử khi phân tích báo cáo quá khứ.
- **Giải pháp Kiến trúc Data Warehouse BigQuery**:
  - Áp dụng **SCD Type 2** cho bảng `dim_products_scd2` để **lưu vết toàn bộ lịch sử biến động giá**.
  - Tạo khóa thay thế Surrogate Key `product_sk = FARM_FINGERPRINT(CONCAT(product_id, '_', CAST(base_price AS STRING)))`.
  - Thêm các trường hiệu lực thời gian: `valid_from` (Ngày bắt đầu áp dụng giá), `valid_to` (Ngày kết thúc giá, `NULL` nếu là hiện tại), và cờ hiệu lực `is_current` (`TRUE`/`FALSE`).
  - Đảm bảo các giao dịch bán lẻ lịch sử được join chính xác với mức giá thực tế tại thời điểm phát sinh đơn hàng.